## 4.1 Pandas approfondi

Jusqu'ici, vous avez analysé **une seule source** à la fois : un CSV, une table SQLite. Mais le quotidien d'un data analyst, c'est de **croiser plusieurs sources** — les ventes d'un côté, le référentiel produits de l'autre, les taux de TVA ailleurs. pandas sait recoller tout ça.

Bonne nouvelle : votre pattern **`filtre → colonne calculée → agrégation`** reste valable. Les outils de ce chapitre viennent simplement **en amont** de cette chaîne, pour reconstituer la table complète avant de l'analyser.

### 4.1.1 — Combiner deux tables avec `pd.merge` (le JOIN de pandas)

`pd.merge` est l'équivalent **exact** du `JOIN` SQL que vous maîtrisez déjà. Il rapproche deux tables sur une **colonne commune** (la clé), et fusionne leurs colonnes.

> 🔗 **Pont SQL** : `pd.merge(a, b, on="produit")` ≡ `SELECT * FROM a JOIN b ON a.produit = b.produit`
> 📊 **Pont Excel** : c'est l'idée du `RECHERCHEV` — aller chercher une info dans une autre table à partir d'une clé.

Toute la subtilité tient dans le paramètre **`how=`**, qui décide **quelles lignes on garde** quand une clé n'existe que d'un seul côté :

| `how=`    | Lignes conservées                                   | Équivalent SQL   |
|-----------|------------------------------------------------------|------------------|
| `"inner"` | uniquement les clés présentes **dans les deux** tables | `INNER JOIN`     |
| `"left"`  | **toutes** celles de gauche (+ les correspondances)  | `LEFT JOIN`      |
| `"right"` | **toutes** celles de droite (+ les correspondances)  | `RIGHT JOIN`     |
| `"outer"` | **toutes** celles des deux côtés                     | `FULL OUTER JOIN`|

Quand une clé manque d'un côté, pandas remplit les colonnes absentes avec **`NaN`** (l'équivalent du `NULL` SQL).

**Ce que fait la cellule suivante** : on crée deux mini-tables « jouet » de 4 lignes chacune — assez petites pour **voir** l'effet de chaque jointure. La table de gauche contient un produit (`Webcam`) absent du référentiel ; la table de droite contient un produit (`Tapis`) jamais vendu. Ces deux « orphelins » rendent visibles les différences entre les 4 types.


In [1]:
import pandas as pd

# =====================================================
# Deux mini-tables "jouet" pour VOIR ce que fait chaque jointure.
# Volontairement minuscules (4 lignes) pour rester lisibles d'un coup d'oeil.
# =====================================================

# Table de GAUCHE : des ventes (produit + quantite)
ventes_jouet = pd.DataFrame({
    "produit":  ["Clavier", "Souris", "Casque", "Webcam"],
    "quantite": [2, 5, 1, 3],
})

# Table de DROITE : un referentiel de prix de revient.
# /!\ "Webcam" est absent ici, et "Tapis" n'a jamais ete vendu -> 2 orphelins
ref_jouet = pd.DataFrame({
    "produit":      ["Clavier", "Souris", "Casque", "Tapis"],
    "prix_revient": [41.0, 8.5, 55.0, 4.0],
})

print("=== Table de GAUCHE : ventes_jouet ===")
print(ventes_jouet)
print("\n=== Table de DROITE : ref_jouet ===")
print(ref_jouet)

# =====================================================
# Les 4 types de jointure, via le parametre how=
# on="produit" : la colonne cle commune (le ON du SQL)
# =====================================================
for type_jointure in ["inner", "left", "right", "outer"]:
    print(f"\n--- how='{type_jointure}' ---")
    print(pd.merge(ventes_jouet, ref_jouet, on="produit", how=type_jointure))

# test d'un inner join dans une variable
resultat = pd.merge(ventes_jouet, ref_jouet, on="produit", how="inner")
print(f"""
Affichage de resultat avec inner
{resultat}
""")


=== Table de GAUCHE : ventes_jouet ===
   produit  quantite
0  Clavier         2
1   Souris         5
2   Casque         1
3   Webcam         3

=== Table de DROITE : ref_jouet ===
   produit  prix_revient
0  Clavier          41.0
1   Souris           8.5
2   Casque          55.0
3    Tapis           4.0

--- how='inner' ---
   produit  quantite  prix_revient
0  Clavier         2          41.0
1   Souris         5           8.5
2   Casque         1          55.0

--- how='left' ---
   produit  quantite  prix_revient
0  Clavier         2          41.0
1   Souris         5           8.5
2   Casque         1          55.0
3   Webcam         3           NaN

--- how='right' ---
   produit  quantite  prix_revient
0  Clavier       2.0          41.0
1   Souris       5.0           8.5
2   Casque       1.0          55.0
3    Tapis       NaN           4.0

--- how='outer' ---
   produit  quantite  prix_revient
0   Casque       1.0          55.0
1  Clavier       2.0          41.0
2   Souris      

### 4.1.2 — `merge` en situation réelle : enrichir les ventes et calculer la marge

Le mini-exemple jouet, c'était pour comprendre. Passons au **vrai cas métier**.

Vos 150 ventes (`ventes_retours.csv`) contiennent le **prix de vente**, mais pas le **prix de revient** ni le **fournisseur** — ces informations vivent dans un **référentiel produits** séparé. Impossible de calculer une marge sans rapprocher les deux tables.

C'est exactement le rôle du `merge` : il vient **en amont** de votre chaîne habituelle. Votre pattern devient :

```
merge   →   filtre   →   colonne calculée   →   agrégation
(NOUVEAU)   (connu)        (connu)               (connu)
```

Le merge **débloque de nouvelles dimensions d'analyse** : la colonne `fournisseur` n'existe nulle part dans vos ventes. En la « branchant » via le référentiel, vous pouvez soudain analyser la marge **par fournisseur** — une question impossible avant la jointure.

**Ce que fait la cellule de code :**

1. On crée le **référentiel produits** (les 12 produits, leur `prix_revient` et leur `fournisseur`). *Ici on le tape à la main ; en 4.2, on ira lire ce même référentiel depuis Snowflake.*
2. On filtre les ventes, puis on les **enrichit** par un `LEFT JOIN` sur la colonne `produit` (on ne veut perdre aucune vente).
3. On calcule la **marge** de chaque vente : `(prix_vente − prix_revient) × quantité`.
4. On **agrège la marge par fournisseur** — la nouvelle dimension offerte par le merge.

> ⚙️ **Pourquoi `left` et pas `inner` ici ?** On part de NOS ventes et on ne veut en perdre aucune. Si un produit vendu manquait au référentiel, un `inner` le ferait disparaître silencieusement du chiffre. Le `left` le garderait (avec un `NaN` bien visible à corriger). Réflexe de prudence.


In [2]:
import pandas as pd

# Rechargement (cellule autonome)
df = pd.read_csv("ventes_retours.csv")

# =====================================================
# 1. Le referentiel produits (tape a la main ici).
#    En 4.2, ce meme referentiel viendra de Snowflake.
# =====================================================
referentiel = pd.DataFrame({
    "produit": ["Casque", "Clavier", "Disque externe", "Enceinte", "Hub USB", "Imprimante",
                "Micro USB", "Souris", "Support écran", "Tapis", "Webcam", "Écran"],
    "prix_revient": [28.0, 41.0, 45.0, 52.0, 6.5, 180.0, 22.0, 8.5, 12.0, 4.0, 19.0, 140.0],
    "fournisseur": ["NordImport", "LogiPro", "NordImport", "TechDirect", "LogiPro", "TechDirect",
                    "LogiPro", "LogiPro", "NordImport", "NordImport", "TechDirect", "TechDirect"],
})

# 2. Filtrer les ventes, puis ENRICHIR via un LEFT JOIN sur "produit"
ventes = df[df["statut"] == "Vente"].copy()
ventes = pd.merge(ventes, referentiel, on="produit", how="left")

# 3. Colonne calculee : marge = (prix de vente - prix de revient) x quantite
ventes["marge"] = (ventes["prix_unitaire"] - ventes["prix_revient"]) * ventes["quantite"]

# 4. Agregation : marge totale par fournisseur (dimension venue du merge)
marge_fournisseur = (
    ventes.groupby("fournisseur")
    .agg(marge_totale=("marge", "sum"), nb_ventes=("marge", "count"))
    .sort_values("marge_totale", ascending=False)
)

print("--- Aperçu des ventes enrichies (5 premières lignes) ---")
print(ventes[["produit", "prix_unitaire", "prix_revient", "quantite", "marge", "fournisseur"]].head())
print("\n--- Marge totale par fournisseur ---")
print(marge_fournisseur)


--- Aperçu des ventes enrichies (5 premières lignes) ---
          produit  prix_unitaire  prix_revient  quantite  marge fournisseur
0          Souris           19.9           8.5         5   57.0     LogiPro
1  Disque externe           69.0          45.0         1   24.0  NordImport
2          Webcam           34.9          19.0         2   31.8  TechDirect
3           Tapis            9.9           4.0         5   29.5  NordImport
4          Souris           19.9           8.5         1   11.4     LogiPro

--- Marge totale par fournisseur ---
             marge_totale  nb_ventes
fournisseur                         
LogiPro            2532.9         54
TechDirect         1889.8         25
NordImport         1534.5         49


### 4.1.3 — `pd.concat` : empiler des sources (et le piège `.append()`)

Le `merge` rapproche deux tables **côte à côte** (il ajoute des colonnes). `pd.concat` fait l'inverse : il **empile** des tables **l'une sous l'autre** (il ajoute des lignes).

C'est le besoin le plus courant du quotidien : vous recevez vos données **en plusieurs morceaux** — un fichier par mois, un export par région, un par magasin — et vous devez les recoller en un seul tableau avant de l'analyser.

> 🔗 **Pont SQL** : `pd.concat([a, b])` ≡ `SELECT * FROM a UNION ALL SELECT * FROM b`. On empile, on ne déduplique pas (pour dédupliquer, il y aura `drop_duplicates`, vu en 4.4).
> 📊 **Pont Excel** : c'est l'équivalent de copier les lignes d'un onglet à la suite d'un autre.

Deux paramètres à connaître :

- **`axis=0`** (le défaut) : on empile des **lignes** — c'est 95 % des cas. `axis=1` empilerait des **colonnes** (plus rare, et risqué si les lignes ne correspondent pas).
- **`ignore_index=True`** : on **renumérote** l'index de 0 à N. Sans lui, les index d'origine se chevauchent (deux lignes « 0 », deux lignes « 1 »…), ce qui pose problème ensuite.

> ⚠️ **Changement important en pandas 3 — à retenir absolument.** L'ancienne méthode `df.append(autre)` (qu'on trouve dans beaucoup de tutos en ligne) a été **supprimée**. Elle ne fonctionne plus du tout. La seule façon correcte d'empiler aujourd'hui, c'est **`pd.concat([...])`**. Si vous voyez `.append()` dans un exemple sur internet, c'est qu'il est périmé.

**Ce que fait la cellule de code** : on simule deux exports reçus séparément (les ventes de janvier, puis celles de février), puis on les empile avec `pd.concat` pour reconstituer un tableau unique.


In [3]:
import pandas as pd

# Rechargement (cellule autonome)
df = pd.read_csv("ventes_retours.csv", parse_dates=["date"])

# =====================================================
# On SIMULE deux exports recus separement : janvier et fevrier.
# Dans la vraie vie, ce seraient deux fichiers CSV distincts
# qu'on aurait charges avec deux pd.read_csv().
# =====================================================
janvier = df[df["date"].dt.month == 1].copy()
fevrier = df[df["date"].dt.month == 2].copy()

print("janvier :", janvier.shape)
print("fevrier :", fevrier.shape)

# =====================================================
# EMPILER les deux tableaux l'un sous l'autre avec pd.concat.
#   axis=0          -> on empile des LIGNES (c'est le defaut)
#   ignore_index=True -> renumerote l'index de 0 a N proprement
#   (sinon les index 0..41 et 0..58 se chevaucheraient)
# Rappel pandas 3 : .append() n'existe plus -> on utilise pd.concat
# =====================================================
trimestre_partiel = pd.concat([janvier, fevrier], ignore_index=True)

print("empilé  :", trimestre_partiel.shape)
print(f"\nVérif : {len(janvier)} + {len(fevrier)} = {len(trimestre_partiel)}")
print("\n--- 5 dernières lignes (index renuméroté de 0 à N) ---")
print(trimestre_partiel.tail()[["id_commande", "date", "produit"]])


janvier : (42, 8)
fevrier : (59, 8)
empilé  : (101, 8)

Vérif : 42 + 59 = 101

--- 5 dernières lignes (index renuméroté de 0 à N) ---
     id_commande       date         produit
96            97 2026-02-27  Disque externe
97            98 2026-02-27       Micro USB
98            99 2026-02-28   Support écran
99           100 2026-02-28          Souris
100          101 2026-02-28          Souris


### 4.1.4 — Dates avancées : `to_datetime`, l'accesseur `.dt` et les intervalles

Vous avez déjà croisé les dates au chapitre 3 (le `parse_dates` de Q5). On approfondit ici, parce que les dates sont **partout** en analyse data, et que pandas a une vraie boîte à outils pour elles.

Le point de départ obligatoire : **une date doit être une vraie date, pas du texte.** Quand pandas lit un CSV, il prend `"2026-02-15"` pour une simple **chaîne de caractères**. Sur du texte, impossible de calculer un écart, d'extraire un mois ou de trier chronologiquement de façon fiable.

`pd.to_datetime(...)` convertit cette colonne texte en type **`datetime`** — une vraie date manipulable.

> 🔗 **Pont SQL** : c'est l'équivalent d'un `CAST(date AS DATE)` suivi des fonctions `YEAR()`, `MONTH()`, `DATEDIFF()` que vous connaissez.

Une fois la colonne convertie, l'accesseur **`.dt`** ouvre toute la boîte à outils :

- `.dt.year` → l'année
- `.dt.month` → le numéro du mois (1 à 12)
- `.dt.day_name()` → le nom du jour de la semaine
- …et bien d'autres (`.dt.day`, `.dt.quarter`, `.dt.dayofweek`…)

Enfin, **soustraire deux dates** donne un **`Timedelta`** (une durée). `date_max − date_min` vous dit l'amplitude couverte par vos données — pratique pour vérifier qu'un export couvre bien la période attendue.

> ⚙️ **À noter en pandas 3** : avant conversion, le type de la colonne est `str` (chaîne) ; après, il devient `datetime64[us]` (`us` = précision à la microseconde). Peu importe l'unité — l'essentiel est que ce soit devenu une vraie date.

**Ce que fait la cellule de code** : convertir la colonne `date`, en extraire année/mois/jour de semaine, puis mesurer l'amplitude du jeu de données (première vente → dernière vente).


In [4]:
import pandas as pd

# Rechargement (cellule autonome)
df = pd.read_csv("ventes_retours.csv")

# =====================================================
# 1. CONVERTIR la colonne texte en vraie date (datetime)
# =====================================================
print("Type AVANT conversion :", df["date"].dtype)
df["date"] = pd.to_datetime(df["date"])
print("Type APRES conversion :", df["date"].dtype)

# =====================================================
# 2. EXTRAIRE des composants avec l'accesseur .dt
# =====================================================
df["annee"] = df["date"].dt.year
df["mois"]  = df["date"].dt.month
df["jour_semaine"] = df["date"].dt.day_name()   # nom du jour (en anglais)

print("\n--- Composants extraits (3 premières lignes) ---")
print(df[["date", "annee", "mois", "jour_semaine"]].head(3))

# =====================================================
# 3. INTERVALLE entre deux dates -> un Timedelta (une duree)
# =====================================================
amplitude = df["date"].max() - df["date"].min()
print(f"\nPremière vente : {df['date'].min().date()}")
print(f"Dernière vente : {df['date'].max().date()}")
print(f"Amplitude      : {amplitude.days} jours")


Type AVANT conversion : str
Type APRES conversion : datetime64[us]

--- Composants extraits (3 premières lignes) ---
        date  annee  mois jour_semaine
0 2026-01-02   2026     1       Friday
1 2026-01-04   2026     1       Sunday
2 2026-01-04   2026     1       Sunday

Première vente : 2026-01-02
Dernière vente : 2026-03-31
Amplitude      : 88 jours


## 4.2 Se connecter à Snowflake

Au chapitre 3.6, vous avez piloté une base **SQLite** — une base **locale**, un simple fichier sur votre disque. **Snowflake**, c'est le même principe, mais une base **professionnelle hébergée dans le cloud** : elle peut contenir des milliards de lignes, plusieurs équipes l'interrogent en même temps, et elle vit sur des serveurs distants.

La **très bonne nouvelle** : côté Python, **presque rien ne change**. Vous utiliserez la même méthode `pd.read_sql_query()` qu'avec SQLite. **Seule la connexion diffère** : au lieu d'ouvrir un fichier, on s'authentifie auprès d'un serveur distant.

### 4.2.1 — Établir la connexion

Se connecter à Snowflake demande quelques **paramètres d'identification** :

- **`account`** : l'identifiant de votre compte Snowflake (fourni par le formateur)
- **`user`** : votre identifiant stagiaire (`STAGIAIRE_1` à `STAGIAIRE_10`)
- **`password`** : le mot de passe — qu'on ne tape **jamais en clair dans le code**
- **`warehouse` / `database` / `schema`** : « où » travailler une fois connecté
- **`role`** : vos droits d'accès (`FORMATION_STAGIAIRE`)

> 🔒 **Sécurité — réflexe pro indispensable.** On n'écrit **jamais** un mot de passe en dur dans une cellule (n'importe qui lisant le notebook le verrait). On utilise `getpass.getpass()` : une petite invite apparaît sous la cellule, vous tapez votre mot de passe, et il **n'apparaît nulle part** dans le code ni dans les sorties.

Après connexion, on lance une **requête de contrôle** (`SELECT CURRENT_VERSION()`) : si elle renvoie un numéro de version, c'est que tout fonctionne.

> ⚙️ **Deux cas de connexion dans la cellule.** Le **Cas A** (actif) est le compte de formation (identifiant + mot de passe). Le **Cas B** (en commentaire) est prêt pour votre futur Snowflake **Cdiscount** en SSO d'entreprise : il suffira de commenter le Cas A et de décommenter le Cas B (`authenticator="externalbrowser"` ouvre alors le navigateur pour l'authentification d'entreprise).

> 🔗 **Dépendance** : cette cellule crée l'objet `conn` (la connexion). **Toutes les cellules Snowflake suivantes le réutilisent** — exécutez donc celle-ci en premier, et ne fermez la connexion qu'à la toute fin (4.2.4).


In [5]:
import snowflake.connector
import getpass
import pandas as pd
import warnings
# Snowflake utilise une connexion DBAPI2 standard : pandas la gere tres bien,
# on masque juste l'avertissement "non teste" pour garder des sorties propres.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")


# =====================================================
# CAS A — Compte de FORMATION (actif par defaut)
# =====================================================
conn = snowflake.connector.connect(
    account="LAWHABL-JB80530",
    user="STAGIAIRE_1",          # <-- remplace par TON numero de stagiaire
    password=getpass.getpass("Mot de passe Snowflake : "),  # saisie masquee
    warehouse="COMPUTE_WH",
    database="FORMATION_DB",
    schema="PUBLIC",
    role="FORMATION_STAGIAIRE",
)

# =====================================================
# CAS B — Snowflake CDISCOUNT (SSO entreprise, APRES la formation)
# Pour l'activer : commente le bloc CAS A ci-dessus, decommente celui-ci,
# et remplace les valeurs entre <>.
# =====================================================
# conn = snowflake.connector.connect(
#     account="<compte_cdiscount>",
#     user="<ton_identifiant_cdiscount>",
#     authenticator="externalbrowser",   # ouvre le navigateur pour le SSO
#     warehouse="<warehouse>",
#     database="<database>",
#     schema="<schema>",
# )

# =====================================================
# Requete de controle : la connexion fonctionne-t-elle ?
# =====================================================
controle = pd.read_sql_query("SELECT CURRENT_VERSION() AS version", conn)
print("Connexion Snowflake réussie !")
print(controle)


Connexion Snowflake réussie !
     VERSION
0  10.19.101


### 4.2.2 — Première vraie requête : lire une table avec `pd.read_sql_query`

Voici le moment où tout se connecte. Souvenez-vous : en 4.1.2, vous aviez tapé un **référentiel produits à la main** dans un `pd.DataFrame`, en notant « *dans la vraie vie, il serait dans une base* ». **Eh bien le voici, pour de vrai, dans Snowflake** : la table `CATALOGUE_PRODUITS`.

Et le geste pour la lire ? **Exactement le même qu'avec SQLite au chapitre 3.6** :

```python
pd.read_sql_query("SELECT * FROM ma_table", conn)
```

On passe une requête SQL et l'objet de connexion `conn`. pandas exécute la requête côté serveur et vous renvoie un DataFrame. **Seule la connexion a changé** (Snowflake au lieu de SQLite) — la méthode, elle, est identique. C'est toute la promesse tenue : ce que vous savez se transfère.

> ⚠️ **Particularité Snowflake : les colonnes reviennent en MAJUSCULES.** `CATALOGUE_PRODUITS` vous donnera des colonnes nommées `PRODUIT`, `PRIX_REVIENT`, `FOURNISSEUR`… alors que vos données locales utilisent `produit` en minuscules. Pour pouvoir les **merger** sur la clé `produit`, il faut d'abord **harmoniser la casse** — sinon pandas ne reconnaîtra pas que `PRODUIT` et `produit` désignent la même chose. On normalise donc les noms en minuscules avec `catalogue.columns = catalogue.columns.str.lower()`.

**Ce que fait la cellule de code :**

1. Lire `CATALOGUE_PRODUITS` depuis Snowflake (`read_sql_query`).
2. Constater les noms de colonnes en majuscules, puis les passer en minuscules.
3. Réutiliser **exactement le merge de 4.1.2** pour enrichir les ventes locales et recalculer la marge — sauf que le référentiel vient maintenant du cloud.

> 🔗 **Dépendance** : cette cellule réutilise la connexion `conn` créée en 4.2.1. Exécutez 4.2.1 d'abord.


In [6]:
import pandas as pd
# La connexion 'conn' a ete creee en 4.2.1 -> execute cette cellule-la d'abord.

# =====================================================
# 1. Lire une table Snowflake : MEME methode qu'avec SQLite (3.6) !
#    Seule la connexion 'conn' change ; read_sql_query est identique.
# =====================================================
catalogue = pd.read_sql_query("SELECT * FROM CATALOGUE_PRODUITS", conn)

print("Dimensions du catalogue :", catalogue.shape)
print("Colonnes (Snowflake renvoie en MAJUSCULES) :", list(catalogue.columns))
print("\n--- Aperçu (3 lignes) ---")
print(catalogue.head(3))

# =====================================================
# 2. Normaliser les noms de colonnes en minuscules pour coller
#    a nos donnees locales (la cle de merge sera 'produit')
# =====================================================
catalogue.columns = catalogue.columns.str.lower()

# =====================================================
# 3. La suite est EXACTEMENT le merge de 4.1.2 : on enrichit les
#    ventes locales, sauf que le referentiel vient de Snowflake.
# =====================================================
df = pd.read_csv("ventes_retours.csv")
ventes = df[df["statut"] == "Vente"].copy()
ventes = pd.merge(
    ventes,
    catalogue[["produit", "prix_revient", "fournisseur"]],
    on="produit",
    how="left",
)
ventes["marge"] = (ventes["prix_unitaire"] - ventes["prix_revient"]) * ventes["quantite"]

print("\n--- Ventes locales enrichies par le catalogue Snowflake ---")
print(ventes[["produit", "prix_unitaire", "prix_revient", "quantite", "marge", "fournisseur"]].head())


Dimensions du catalogue : (12, 6)
Colonnes (Snowflake renvoie en MAJUSCULES) : ['PRODUIT', 'CATEGORIE', 'PRIX_REVIENT', 'FOURNISSEUR', 'MARGE_CIBLE_PCT', 'DATE_REFERENCEMENT']

--- Aperçu (3 lignes) ---
          PRODUIT CATEGORIE  PRIX_REVIENT FOURNISSEUR  MARGE_CIBLE_PCT  \
0          Casque   Premium          28.0  NordImport             35.0   
1         Clavier  Standard          41.0     LogiPro             25.0   
2  Disque externe   Premium          45.0  NordImport             30.0   

  DATE_REFERENCEMENT  
0         2024-01-15  
1         2023-06-10  
2         2024-03-22  

--- Ventes locales enrichies par le catalogue Snowflake ---
          produit  prix_unitaire  prix_revient  quantite  marge fournisseur
0          Souris           19.9           8.5         5   57.0     LogiPro
1  Disque externe           69.0          45.0         1   24.0  NordImport
2          Webcam           34.9          19.0         2   31.8  TechDirect
3           Tapis            9.9           

### 4.2.3 — 🥇 La règle d'or : filtrer côté serveur, pas dans pandas

Voici **le réflexe le plus important** de tout ce chapitre Snowflake. Il sépare le data analyst débutant du data analyst efficace.

Quand vos données sont dans une base distante, vous avez deux façons d'obtenir un sous-ensemble (par exemple « uniquement les ventes d'Île-de-France ») :

- **❌ La mauvaise** : `SELECT * FROM table` — vous rapatriez **toute** la table sur votre machine, **puis** vous filtrez dans pandas. Snowflake vous envoie des millions de lignes par le réseau, dont 95 % que vous allez jeter.
- **✅ La bonne** : `SELECT * FROM table WHERE region = 'Île-de-France'` — vous demandez à **Snowflake de filtrer**, et il ne vous renvoie que les lignes utiles.

> 🔗 **Vous savez déjà faire ça.** Le `WHERE` SQL, c'est exactement ce que vous maîtrisez depuis le chapitre 3.6. La nouveauté n'est pas technique, c'est un **réflexe** : poussez le maximum de travail (filtres, agrégations, jointures) **côté serveur**, là où la base est puissante, et ne rapatriez que le résultat.

**Pourquoi c'est crucial :**

1. **Le réseau est le goulot d'étranglement.** Transférer 20 000 lignes (ou 20 millions) prend du temps. Transférer 3 000 lignes en prend beaucoup moins.
2. **La mémoire de votre PC est limitée.** `SELECT *` sur une grosse table peut tout simplement **saturer la RAM** de votre poste. Snowflake, lui, est dimensionné pour ça.
3. **Snowflake filtre plus vite que pandas.** C'est une base conçue pour traiter d'énormes volumes en parallèle.

**Ce que fait la cellule de code** : on compare les deux méthodes sur `HISTORIQUE_VENTES` (20 000 lignes), chronomètre à l'appui. On vérifie qu'elles donnent **le même résultat**, mais pas du tout le même volume transféré.


In [7]:
import pandas as pd
import time
# 'conn' : connexion Snowflake creee en 4.2.1 (a executer avant)

# =====================================================
# ❌ MAUVAISE methode : tout charger, PUIS filtrer dans pandas
#    -> Snowflake renvoie les 20 000 lignes par le reseau
# =====================================================
t0 = time.perf_counter()
tout = pd.read_sql_query("SELECT * FROM HISTORIQUE_VENTES", conn)
idf_pandas = tout[tout["CLIENT_REGION"] == "Île-de-France"]
t_pandas = time.perf_counter() - t0

# =====================================================
# ✅ BONNE methode : filtrer cote SERVEUR avec un WHERE
#    -> Snowflake ne renvoie QUE les lignes utiles
# =====================================================
t0 = time.perf_counter()
idf_serveur = pd.read_sql_query(
    "SELECT * FROM HISTORIQUE_VENTES WHERE CLIENT_REGION = 'Île-de-France'", conn
)
t_serveur = time.perf_counter() - t0

# =====================================================
# Comparaison
# =====================================================
print(f"Méthode pandas  : {len(tout):>6} lignes transférées  ({t_pandas:.2f} s)")
print(f"Méthode serveur : {len(idf_serveur):>6} lignes transférées  ({t_serveur:.2f} s)")
print(f"\nMême résultat ? {len(idf_pandas) == len(idf_serveur)}")
print(f"Volume transféré : {len(tout) / len(idf_serveur):.1f}x moins avec le WHERE serveur")


Méthode pandas  :  20000 lignes transférées  (1.33 s)
Méthode serveur :   2422 lignes transférées  (0.17 s)

Même résultat ? True
Volume transféré : 8.3x moins avec le WHERE serveur


### 4.2.4 — Bonnes pratiques de connexion

Vous savez vous connecter et requêter. Reste à le faire **proprement**, comme en entreprise. Quatre réflexes.

**1. Explorer avec `LIMIT`**

Quand vous découvrez une table inconnue, ne rapatriez **jamais** tout pour « voir à quoi ça ressemble ». Un `LIMIT 5` suffit à inspecter la structure et quelques valeurs, sans transférer des millions de lignes. C'est le `df.head()` de pandas, mais appliqué **côté serveur** — dans l'esprit exact de la règle d'or de 4.2.3.

**2. Ne jamais coder les secrets en dur**

Un mot de passe écrit en clair dans une cellule (`password="..."`) finit dans le notebook, dans Git, dans les captures d'écran… On utilise `getpass.getpass()` (saisie au runtime), comme depuis 4.2.1. Pour des scripts automatisés, on irait plus loin : **variables d'environnement** ou **fichier de configuration** hors du code. La règle : le secret ne doit **jamais** vivre dans le code source.

**3. Fermer la connexion quand on a fini**

Une connexion ouverte consomme des ressources côté serveur. On la **ferme** avec `conn.close()` une fois le travail terminé. Cela aide aussi l'**auto-suspend** du warehouse (mise en veille = économies).

**4. Le `with` : la fermeture automatique (réflexe pro)**

La façon la plus sûre de ne jamais oublier de fermer, c'est le **context manager** `with` : la connexion se ferme **toute seule** à la sortie du bloc, même en cas d'erreur. Modèle à réutiliser pour vos futurs scripts :

```python
import snowflake.connector
import getpass
import pandas as pd

with snowflake.connector.connect(
    account="LAWHABL-JB80530",
    user="STAGIAIRE_1",
    password=getpass.getpass("Mot de passe : "),
    warehouse="COMPUTE_WH",
    database="FORMATION_DB",
    schema="PUBLIC",
    role="FORMATION_STAGIAIRE",
) as conn:
    df = pd.read_sql_query("SELECT * FROM CATALOGUE_PRODUITS LIMIT 100", conn)

# Ici, hors du bloc, la connexion est DEJA fermee automatiquement.
print(df.shape)
```

> 🔗 **Pont SQLite** : c'est la même idée de « bien refermer ce qu'on a ouvert » qu'avec les fichiers et les connexions du chapitre 3.6 — le `with` automatise juste la fermeture.

**Ce que fait la cellule de code** : illustrer l'exploration avec `LIMIT`, puis **fermer proprement** la connexion `conn` ouverte en 4.2.1 (c'est la dernière cellule Snowflake de cette section).


In [8]:
import pandas as pd
# 'conn' : connexion creee en 4.2.1.

# =====================================================
# BONNE PRATIQUE 1 — Explorer avec LIMIT (le "head()" cote serveur)
# On ne rapatrie que 5 lignes pour decouvrir la structure de la table.
# =====================================================
apercu = pd.read_sql_query("SELECT * FROM HISTORIQUE_VENTES LIMIT 5", conn)
print("--- Aperçu (5 lignes) de HISTORIQUE_VENTES ---")
print(apercu)

# =====================================================
# BONNE PRATIQUE 2 — Fermer la connexion quand on a termine
# Libere les ressources et favorise l'auto-suspend du warehouse.
# (C'est la derniere cellule Snowflake de 4.2 : on referme le tuyau.)
# =====================================================
conn.close()
print("\nConnexion fermée proprement.")


--- Aperçu (5 lignes) de HISTORIQUE_VENTES ---
   DATE_VENTE        PRODUIT  QUANTITE         CLIENT_REGION  PRIX_UNITAIRE
0  2025-05-24          Écran         3  Auvergne-Rhône-Alpes         159.98
1  2025-08-10  Support écran         3    Nouvelle-Aquitaine         195.87
2  2024-09-13     Imprimante         3             Grand Est         146.32
3  2025-07-25      Micro USB         5      Pays de la Loire          37.95
4  2024-05-20       Enceinte         5    Nouvelle-Aquitaine          18.24

Connexion fermée proprement.


In [9]:
import snowflake.connector
import getpass
import pandas as pd
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

# =====================================================
# Meme exploration qu'en 4.2.4, mais avec un with :
# la connexion s'OUVRE au debut du bloc et se FERME
# automatiquement a la sortie (meme si une erreur survient).
# =====================================================
with snowflake.connector.connect(
    account="LAWHABL-JB80530",
    user="STAGIAIRE_1",          # <-- ton numero de stagiaire
    password=getpass.getpass("Mot de passe Snowflake : "),
    warehouse="COMPUTE_WH",
    database="FORMATION_DB",
    schema="PUBLIC",
    role="FORMATION_STAGIAIRE",
) as conn:
    # /!\ Tout le travail Snowflake se fait DANS le bloc (lignes indentees)
    apercu = pd.read_sql_query("SELECT * FROM HISTORIQUE_VENTES LIMIT 5", conn)
    print("--- Aperçu (5 lignes) de HISTORIQUE_VENTES ---")
    print(apercu)

# Ici, hors du bloc : la connexion est DEJA fermee, sans conn.close().
print("\nConnexion fermée automatiquement (sortie du bloc with).")


--- Aperçu (5 lignes) de HISTORIQUE_VENTES ---
   DATE_VENTE        PRODUIT  QUANTITE         CLIENT_REGION  PRIX_UNITAIRE
0  2025-05-24          Écran         3  Auvergne-Rhône-Alpes         159.98
1  2025-08-10  Support écran         3    Nouvelle-Aquitaine         195.87
2  2024-09-13     Imprimante         3             Grand Est         146.32
3  2025-07-25      Micro USB         5      Pays de la Loire          37.95
4  2024-05-20       Enceinte         5    Nouvelle-Aquitaine          18.24

Connexion fermée automatiquement (sortie du bloc with).


## 4.3 — Lire d'autres sources : Excel et JSON

Jusqu'ici, vos données venaient d'un **CSV** (chapitre 3) ou d'une **base Snowflake** (4.2). Mais dans la vraie vie d'un data analyst, les données arrivent de **partout** : la base centrale, bien sûr, mais aussi le **fichier Excel** que le métier vous envoie par mail, ou la **réponse JSON** d'un service externe.

Savoir lire ces formats, c'est pouvoir **réconcilier** toutes ces sources dans un seul rapport pandas — ce sera précisément l'objet du TP de cette section.

> 🔗 **L'image à garder en tête.** Snowflake (4.2), c'est l'**entrepôt central** de l'entreprise. L'Excel, c'est le fichier que **Loïc vous envoie** (des objectifs commerciaux qui ne sont *pas* dans la base). Le JSON, c'est ce qu'un **partenaire/service externe** expose. Le métier consiste à faire **parler ces sources ensemble**.

### 4.3.1 — Lire un fichier Excel avec `pd.read_excel`

Un classeur Excel (`.xlsx`) peut contenir **plusieurs feuilles** (les onglets en bas de la fenêtre Excel). pandas sait les lire avec une seule fonction, `pd.read_excel`, dont le paramètre clé est **`sheet_name`** :

- `sheet_name="Objectifs"` → lit **une feuille précise** et renvoie **un DataFrame** (comme `read_csv`).
- `sheet_name=None` → lit **toutes les feuilles** d'un coup et renvoie un **dictionnaire** `{ "nom_de_feuille": DataFrame }`. On récupère ensuite chaque feuille par son nom, comme dans un dictionnaire Python.

> 💡 **Analogie Excel directe.** `sheet_name`, c'est l'**onglet** que vous cliquez en bas du classeur. Demander une feuille = cliquer sur un onglet ; demander `None` = ouvrir tout le classeur d'un coup.

Pour cet exercice, on dispose du classeur **`objectifs_commerciaux_2026.xlsx`** (fourni, à côté du notebook). Il contient deux feuilles :
- **`Objectifs`** : l'objectif de chiffre d'affaires du trimestre par catégorie de produits ;
- **`Responsables`** : qui pilote chaque catégorie.

**À toi de jouer.** Lis la feuille `Objectifs` seule, affiche-la. Puis lis **tout le classeur** d'un coup et affiche la liste des feuilles disponibles ainsi que le contenu de la feuille `Responsables`.

> ⚠️ **Pré-requis technique.** La lecture d'Excel s'appuie sur la bibliothèque **`openpyxl`** (déjà installée dans votre environnement). Si un jour `pd.read_excel` renvoie une erreur du type *« Missing optional dependency 'openpyxl' »*, c'est qu'elle manque : `pip install openpyxl`.


In [1]:
import pandas as pd

# Le fichier objectifs_commerciaux_2026.xlsx est fourni, a cote du notebook.

# =====================================================
# 1. Lire UNE feuille precise -> on obtient un DataFrame
#    sheet_name = le nom de l'onglet, exactement comme dans Excel.
# =====================================================
objectifs = pd.read_excel("objectifs_commerciaux_2026.xlsx", sheet_name="Objectifs")
print("--- Feuille 'Objectifs' (un DataFrame) ---")
print(objectifs)
print("Type renvoye :", type(objectifs).__name__)

# =====================================================
# 2. Lire TOUTES les feuilles d'un coup -> on obtient un DICTIONNAIRE
#    {nom_de_feuille: DataFrame}. On pioche ensuite par le nom de l'onglet.
# =====================================================
classeur = pd.read_excel("objectifs_commerciaux_2026.xlsx", sheet_name=None)
print("\n--- Classeur complet (un dictionnaire) ---")
print("Type renvoye :", type(classeur).__name__)
print("Feuilles disponibles :", list(classeur.keys()))

# On recupere une feuille par son nom, comme dans un dictionnaire.
responsables = classeur["Responsables"]
print("\n--- Feuille 'Responsables' ---")
print(responsables)


--- Feuille 'Objectifs' (un DataFrame) ---
         categorie  objectif_ca_t1
0  Entrée de gamme            3000
1         Standard            7500
2          Premium            3500
Type renvoye : DataFrame

--- Classeur complet (un dictionnaire) ---
Type renvoye : dict
Feuilles disponibles : ['Objectifs', 'Responsables']

--- Feuille 'Responsables' ---
         categorie responsable
0  Entrée de gamme     Soumaya
1         Standard      Aurore
2          Premium     Raphaël


### 4.3.2 — Lire un fichier JSON avec `pd.read_json`

Le **JSON** est un format **texte** très répandu pour **échanger des données entre applications**, en particulier sur le web : quand un site, une appli mobile ou un service partenaire vous expose des données via une **API** (une « prise » à laquelle on se branche pour récupérer de l'information), la réponse est presque toujours du JSON.

Sa forme la plus courante est une **liste d'objets**, où chaque objet est un ensemble de paires `"clé": valeur`. Visuellement :

```json
[
  {"produit": "Souris", "note_moyenne": 4.3, "nb_avis": 512},
  {"produit": "Casque", "note_moyenne": 4.6, "nb_avis": 874}
]
```

> 💡 **Le bon réflexe de lecture.** Voyez ce JSON comme un **tableau** : chaque **objet `{...}`** = une **ligne**, chaque **clé** (`produit`, `note_moyenne`, `nb_avis`) = une **colonne**. C'est exactement la structure d'un DataFrame… d'où la simplicité de la conversion.

`pd.read_json("fichier.json")` lit ce fichier et le transforme **directement en DataFrame** : un objet par ligne, une clé par colonne. C'est aussi simple que `read_csv`.

Pour cet exercice, on dispose du fichier **`avis_clients_api.json`** (fourni, à côté du notebook). Il simule la **réponse d'une API d'avis clients** : pour chacun des 12 produits, sa **note moyenne** et son **nombre d'avis**.

> 🔗 **Pourquoi un fichier local et pas un vrai appel réseau ?** Pour que l'exercice fonctionne **partout, tout le temps**, sans dépendre du Wi-Fi ni d'un service externe qui pourrait être indisponible. Le contenu est **identique** à ce qu'une vraie API renverrait — seul le mode de récupération change.

**À toi de jouer.** Lis `avis_clients_api.json`, affiche les premières lignes, puis vérifie ce que pandas a déduit : le **nombre de lignes/colonnes** (`shape`) et le **type de chaque colonne** (`dtypes`).

> 🚀 **Pour aller plus loin — le vrai appel d'API (hors programme).** Dans la vraie vie, on récupérerait le JSON depuis le web avec la bibliothèque `requests` (présente dans votre environnement). Le schéma, à titre d'illustration :
> ```python
> import requests
> reponse = requests.get("https://api.un-service.com/avis")  # une vraie API d'avis
> avis = pd.DataFrame(reponse.json())                        # .json() -> liste d'objets -> DataFrame
> ```
> L'idée à retenir : qu'il vienne d'un **fichier** ou du **réseau**, le JSON finit en **DataFrame** de la même façon. On reste sur le fichier local pour la suite.


In [3]:
import pandas as pd

# Le fichier avis_clients_api.json est fourni, a cote du notebook.

# =====================================================
# Lire un JSON (liste d'objets) -> DataFrame directement.
#   1 objet {...} = 1 ligne ; chaque cle = 1 colonne.
# =====================================================
avis = pd.read_json("avis_clients_api.json")

print("--- Avis clients (5 premieres lignes) ---")
print(avis.head())

# Verifier ce que pandas a construit : dimensions et types de colonnes.
print("\nDimensions (lignes, colonnes) :", avis.shape)
print("\nTypes de chaque colonne :")
print(avis.dtypes)


--- Avis clients (5 premieres lignes) ---
          produit  note_moyenne  nb_avis
0          Souris           4.3      512
1          Casque           4.6      874
2  Disque externe           4.1      203
3          Webcam           3.8      147
4           Tapis           4.4      321

Dimensions (lignes, colonnes) : (12, 3)

Types de chaque colonne :
produit             str
note_moyenne    float64
nb_avis           int64
dtype: object


## 4.4 — Nettoyage rapide : la réalité du métier data

Voici une vérité que tout data analyst apprend vite : **environ 80 % du travail consiste à nettoyer les données**, et seulement 20 % à les analyser. Les données réelles arrivent **sales** : des lignes en **double** (un même enregistrement importé deux fois), des **valeurs manquantes** (une case vide), des **types incohérents** (un prix stocké comme du texte au lieu d'un nombre).

Tant que ces problèmes ne sont pas réglés, **tous vos calculs sont faux** : une somme qui compte deux fois la même vente, une moyenne faussée par des trous, une multiplication qui plante parce que le prix est du texte. Cette section présente les **trois outils de nettoyage rapide** que vous emploierez quotidiennement.

Pour les voir à l'œuvre, on part d'une **petite table d'exemple volontairement « sale »** — un mini-extrait de commandes tel qu'il pourrait sortir d'un import bancal :

```
   id_commande  produit  quantite prix_unitaire  client_region
0         1001   Souris       2.0         12.90  Île-de-France
1         1002   Casque       1.0         79.00       Bretagne
2         1002   Casque       1.0         79.00       Bretagne   <- ligne en double
3         1003  Clavier       3.0         29.90            NaN   <- region manquante
4         1004   Webcam       NaN         45.00      Occitanie   <- quantite manquante
5         1005    Tapis       2.0          9.90      Grand Est
```

Trois défauts y sont cachés exprès : **un doublon** (la ligne 1002), **deux valeurs manquantes** (`NaN`), et la colonne `prix_unitaire` qui est en **texte** et non en nombre. On va les corriger un par un. Cette même table `commandes_brutes` servira de fil rouge à toute la section 4.4.

### 4.4.1 — Supprimer les doublons avec `drop_duplicates`

Une ligne **en double**, c'est un enregistrement strictement identique à un autre — typiquement un fichier importé deux fois, ou une fusion de sources qui se chevauchent. Si on ne l'enlève pas, il **gonfle artificiellement** les comptes et les sommes.

Deux gestes :
- `df.duplicated()` → renvoie une Série de `True`/`False` (`True` = « cette ligne est un doublon d'une ligne déjà vue »). `df.duplicated().sum()` en donne le **nombre**.
- `df.drop_duplicates()` → renvoie le DataFrame **sans les doublons** (il garde la **première** occurrence).

> 💡 **Analogies que vous connaissez.** `drop_duplicates`, c'est le **`SELECT DISTINCT`** du SQL, ou le bouton **« Supprimer les doublons »** d'Excel (onglet Données). Même idée : ne garder qu'un exemplaire de chaque ligne identique.

**À toi de jouer.** Compte d'abord combien de doublons contient `commandes_brutes`, puis crée une version nettoyée `commandes` sans doublons (avec le réflexe `.copy()`). Vérifie le nombre de lignes avant/après.


In [4]:
import pandas as pd
import numpy as np

# =====================================================
# La table "sale" fil rouge de la section 4.4.
# (On la cree ici ; les sous-sections 4.4.2 et 4.4.3 la reutilisent.)
# =====================================================
commandes_brutes = pd.DataFrame({
    "id_commande":   [1001, 1002, 1002, 1003, 1004, 1005],
    "produit":       ["Souris", "Casque", "Casque", "Clavier", "Webcam", "Tapis"],
    "quantite":      [2, 1, 1, 3, np.nan, 2],            # np.nan = une quantite manquante
    "prix_unitaire": ["12.90", "79.00", "79.00", "29.90", "45.00", "9.90"],  # /!\ du TEXTE
    "client_region": ["Île-de-France", "Bretagne", "Bretagne", None, "Occitanie", "Grand Est"],
})
print("--- Table brute (6 lignes) ---")
print(commandes_brutes)

# =====================================================
# 4.4.1 — Reperer puis supprimer les doublons
# =====================================================
print("\nNombre de lignes en double :", commandes_brutes.duplicated().sum())

# drop_duplicates garde la 1re occurrence ; .copy() = reflexe defensif apres filtre.
commandes = commandes_brutes.drop_duplicates().copy()

print("Lignes avant :", len(commandes_brutes), "| apres :", len(commandes))
print("\n--- Sans doublon (5 lignes) ---")
print(commandes)


--- Table brute (6 lignes) ---
   id_commande  produit  quantite prix_unitaire  client_region
0         1001   Souris       2.0         12.90  Île-de-France
1         1002   Casque       1.0         79.00       Bretagne
2         1002   Casque       1.0         79.00       Bretagne
3         1003  Clavier       3.0         29.90            NaN
4         1004   Webcam       NaN         45.00      Occitanie
5         1005    Tapis       2.0          9.90      Grand Est

Nombre de lignes en double : 1
Lignes avant : 6 | apres : 5

--- Sans doublon (5 lignes) ---
   id_commande  produit  quantite prix_unitaire  client_region
0         1001   Souris       2.0         12.90  Île-de-France
1         1002   Casque       1.0         79.00       Bretagne
3         1003  Clavier       3.0         29.90            NaN
4         1004   Webcam       NaN         45.00      Occitanie
5         1005    Tapis       2.0          9.90      Grand Est


### 4.4.2 — Gérer les valeurs manquantes : `isna`, `fillna`, `dropna`

Une **valeur manquante** (`NaN` pour un nombre, `<NA>` pour du texte) est une case **vide** : la donnée n'a pas été saisie, ou s'est perdue à l'import. C'est le problème le plus fréquent du nettoyage, et le plus traître : une seule case vide peut faire **planter un calcul** (une multiplication avec `NaN` donne `NaN`) ou **fausser une moyenne**.

Trois outils, dans l'ordre logique **repérer → décider → agir** :

- **Repérer** : `df.isna()` renvoie un tableau de `True`/`False` (`True` = case vide). En pratique, `df.isna().sum()` donne le **nombre de trous par colonne** — votre premier diagnostic.
- **Combler** : `df["colonne"].fillna(valeur)` **remplace** les cases vides par une valeur de votre choix (0, une moyenne, `"Inconnue"`…).
- **Supprimer** : `df.dropna()` **supprime les lignes** qui contiennent au moins une case vide.

> 💡 **Analogies que vous connaissez.** Une case vide, c'est le **`NULL`** du SQL. `fillna`, c'est exactement le **`COALESCE(colonne, valeur)`** SQL (« si c'est vide, mets ça à la place »). Et `dropna`, c'est filtrer pour **enlever les lignes vides**, comme un `WHERE colonne IS NOT NULL` ou un tri/filtre des cellules vides sous Excel.

**Combler ou supprimer ? La décision vous appartient** — c'est un choix **métier**, pas technique :
- Une `quantite` manquante peut raisonnablement valoir **0** (`fillna(0)`).
- Une `region` manquante peut devenir **`"Inconnue"`** plutôt que de jeter toute la commande.
- Mais si une ligne est **trop incomplète** pour être exploitable, mieux vaut la **supprimer** (`dropna`).

**À toi de jouer.** Sur la table `commandes` (dédoublonnée en 4.4.1) : compte les manquants par colonne, puis produis une version **comblée** (`quantite`→0, `region`→`"Inconnue"`). Affiche aussi, pour comparaison, ce que donnerait la **suppression** des lignes incomplètes.


In [5]:
import pandas as pd
# commandes : table SANS doublon, creee en 4.4.1 -> execute cette cellule-la d'abord.

# =====================================================
# 1. REPERER : combien de cases vides, et dans quelles colonnes ?
# =====================================================
print("--- Valeurs manquantes par colonne ---")
print(commandes.isna().sum())

# =====================================================
# 2. COMBLER (fillna) : on remplit selon une logique METIER, colonne par colonne.
#    quantite manquante -> 0 ; region manquante -> "Inconnue".
# =====================================================
commandes_pleines = commandes.copy()
commandes_pleines["quantite"] = commandes_pleines["quantite"].fillna(0)
commandes_pleines["client_region"] = commandes_pleines["client_region"].fillna("Inconnue")
print("\n--- Apres fillna (quantite->0, region->'Inconnue') ---")
print(commandes_pleines)

# =====================================================
# 3. ALTERNATIVE (dropna) : supprimer les lignes incompletes plutot que combler.
# =====================================================
commandes_completes = commandes.dropna().copy()
print("\n--- Apres dropna (uniquement les lignes 100% completes) ---")
print("Lignes restantes :", len(commandes_completes))
print(commandes_completes)


--- Valeurs manquantes par colonne ---
id_commande      0
produit          0
quantite         1
prix_unitaire    0
client_region    1
dtype: int64

--- Apres fillna (quantite->0, region->'Inconnue') ---
   id_commande  produit  quantite prix_unitaire  client_region
0         1001   Souris       2.0         12.90  Île-de-France
1         1002   Casque       1.0         79.00       Bretagne
3         1003  Clavier       3.0         29.90       Inconnue
4         1004   Webcam       0.0         45.00      Occitanie
5         1005    Tapis       2.0          9.90      Grand Est

--- Apres dropna (uniquement les lignes 100% completes) ---
Lignes restantes : 3
   id_commande produit  quantite prix_unitaire  client_region
0         1001  Souris       2.0         12.90  Île-de-France
1         1002  Casque       1.0         79.00       Bretagne
5         1005   Tapis       2.0          9.90      Grand Est


### 4.4.3 — Corriger les types avec `astype`

Chaque colonne d'un DataFrame a un **type** (`dtype`) : `int64` (entiers), `float64` (décimaux), `str` (texte), `datetime64` (dates)… Le problème classique : une colonne qui **ressemble** à des nombres mais que pandas a chargée comme du **texte** (parce qu'un fichier source la stockait entre guillemets, par exemple).

C'est insidieux, car **les calculs ne plantent pas forcément** — ils donnent un résultat **absurde, en silence** :
- `"12.90" * 2` ne donne pas `25.80`, mais **`"12.9012.90"`** (Python *répète* le texte) ;
- `somme` d'une colonne de texte ne fait pas l'addition, elle **colle les morceaux bout à bout** (`"12.9079.00..."`).

La correction : **`df["colonne"] = df["colonne"].astype(float)`** convertit le texte en vrais nombres. On réaffecte la colonne pour que le changement soit conservé.

> 💡 **Analogies que vous connaissez.** `astype`, c'est le **`CAST`** / **`CONVERT`** du SQL (`CAST(prix AS FLOAT)`). Sous Excel, c'est le cas du **nombre stocké en texte** — la petite alerte triangle vert « Ce nombre est au format texte » et l'option *« Convertir en nombre »*. Même problème, même solution.

> ⚠️ **Quand `astype(float)` ne suffit pas.** Il exige un texte **propre** avec un point décimal (`"12.90"`). Si le texte contient une **virgule** décimale (`"12,90"`, format français) ou un symbole (`"12,90 €"`), `astype(float)` **échoue**. Il faut alors d'abord nettoyer le texte (ex. remplacer la virgule par un point), ou utiliser `pd.to_numeric(..., errors="coerce")`, plus tolérant. À garder en tête pour les vrais fichiers.

**À toi de jouer.** Sur la table `commandes` (de 4.4.1) : constate le type de `prix_unitaire`, illustre le piège du calcul sur du texte, puis convertis la colonne en `float` et vérifie qu'un calcul redevient correct.


In [6]:
import pandas as pd
# commandes : table dedoublonnee creee en 4.4.1 -> execute cette cellule-la d'abord.

# =====================================================
# 1. CONSTAT : 'prix_unitaire' est du TEXTE, pas un nombre.
# =====================================================
print("Type de 'prix_unitaire' AVANT :", commandes["prix_unitaire"].dtype)

# =====================================================
# 2. LE PIEGE : un calcul sur du texte ne plante pas, il donne n'importe quoi.
#    "12.90" * 2 ne fait pas 25.80 -> il REPETE le texte ("12.9012.90").
# =====================================================
print("\nprix * 2 sur du TEXTE (resultat absurde) :")
print((commandes["prix_unitaire"] * 2).tolist())

# =====================================================
# 3. LA CORRECTION : convertir en nombre reel avec astype(float).
#    On reaffecte la colonne pour conserver le changement.
# =====================================================
commandes_typees = commandes.copy()
commandes_typees["prix_unitaire"] = commandes_typees["prix_unitaire"].astype(float)
print("\nType de 'prix_unitaire' APRES :", commandes_typees["prix_unitaire"].dtype)

# =====================================================
# 4. Maintenant, les calculs sont JUSTES.
# =====================================================
print("prix * 2 sur des NOMBRES (correct) :")
print((commandes_typees["prix_unitaire"] * 2).tolist())
print("Somme des prix :", round(commandes_typees["prix_unitaire"].sum(), 2), "EUR")


Type de 'prix_unitaire' AVANT : str

prix * 2 sur du TEXTE (resultat absurde) :
['12.9012.90', '79.0079.00', '29.9029.90', '45.0045.00', '9.909.90']

Type de 'prix_unitaire' APRES : float64
prix * 2 sur des NOMBRES (correct) :
[25.8, 158.0, 59.8, 90.0, 19.8]
Somme des prix : 176.7 EUR


> 🚀 **Pour aller plus loin — forcer l'arrondi « scolaire » (façon Excel/SQL)**
>
> Le `round()` de Python applique l'**arrondi du banquier** : sur un `.5` exact, il va vers l'entier **pair** (`round(2.5)` → `2`, `round(14.5)` → `14`). C'est volontaire (ça évite un biais quand on somme beaucoup de valeurs), mais **Excel et SQL**, eux, font l'arrondi **« .5 toujours vers le haut »** (`ARRONDI(2,5;0)` → `3`). D'où des écarts d'un centime quand on recoupe un reporting pandas avec un fichier Excel.
>
> Si vous devez **coller au comportement Excel**, passez par le module `decimal` avec la règle `ROUND_HALF_UP`. On encapsule ça dans une petite fonction réutilisable. Astuce importante : on convertit le nombre via `str(x)` **avant** de le passer à `Decimal`, pour neutraliser au passage l'imprécision des flottants (le fameux `2.675` stocké comme `2.6749999…`).


In [7]:
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP
# commandes_typees : table nettoyee creee en 4.4.3 (prix en float) -> execute 4.4.3 d'abord.

# =====================================================
# Fonction d'arrondi SCOLAIRE (.5 toujours vers le haut), facon Excel/SQL.
#   - n = nombre de decimales voulues.
#   - str(x) avant Decimal -> evite l'imprecision des flottants.
# =====================================================
def arrondi_scolaire(x, n=2):
    gabarit = Decimal("1." + "0" * n) if n > 0 else Decimal("1")
    return float(Decimal(str(x)).quantize(gabarit, rounding=ROUND_HALF_UP))

# Memes donnees qu'en 4.4.3 : on arrondit la somme des prix.
total = commandes_typees["prix_unitaire"].sum()

print("Somme des prix       :", total)
print("round(total, 2)      :", round(total, 2), "  <- arrondi du banquier (defaut Python)")
print("arrondi_scolaire(...) :", arrondi_scolaire(total, 2), "  <- arrondi scolaire (facon Excel)")

# Demonstration sur des valeurs pile a .5 : la difference saute aux yeux.
print("\n--- Comparaison sur des valeurs pile a .5 ---")
for v in [0.5, 2.5, 14.5]:
    print(f"{v} -> round (banquier) = {round(v)}   |   scolaire = {arrondi_scolaire(v, 0)}")


Somme des prix       : 176.70000000000002
round(total, 2)      : 176.7   <- arrondi du banquier (defaut Python)
arrondi_scolaire(...) : 176.7   <- arrondi scolaire (facon Excel)

--- Comparaison sur des valeurs pile a .5 ---
0.5 -> round (banquier) = 0   |   scolaire = 1.0
2.5 -> round (banquier) = 2   |   scolaire = 3.0
14.5 -> round (banquier) = 14   |   scolaire = 15.0


## 4.5 — TP guidé : Reporting multi-sources 🎯

C'est l'aboutissement du chapitre : faire parler **ensemble** une base **Snowflake**, un **Excel** et un **JSON**, pour produire un seul tableau de pilotage. Pas d'inquiétude : on avance **étape par étape**, et chaque ligne à écrire t'est indiquée. Tu as déjà vu toutes ces briques séparément (4.1 à 4.4) — ici, on les assemble.

**Le contexte.** Tu es analyste pricing. On te demande, pour chaque **catégorie** de produits, un tableau qui répond à : *le CA réalisé atteint-il l'objectif ? qui en est responsable ? les clients sont-ils satisfaits ?*

**Le résultat à obtenir** (un DataFrame nommé `rapport`) :

```
         categorie  ca_realise  objectif_ca_t1 responsable  note_moyenne  taux_atteinte_pct   statut
0  Entrée de gamme  2151598.95         2000000     Soumaya          4.10              107.6  Atteint
1         Standard  1570514.56         1600000      Aurore          4.07               98.2   Manqué
2          Premium  2580704.37         2700000     Raphaël          4.22               95.6   Manqué
```

On y va doucement, dans cet ordre. **Écris chaque bloc dans la cellule de code, l'un après l'autre.**

---

### Étape 0 — Les outils (à mettre tout en haut)

Comme d'habitude, on importe ce dont on a besoin. Recopie :
```python
import snowflake.connector
import getpass
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
```
👉 `numpy` (surnommé `np`) est nouveau ici : il nous servira **uniquement** à l'étape 5 pour la colonne `statut`.

---

### Étape 1 — Le CA réalisé, depuis Snowflake 🗄️

**But :** récupérer le chiffre d'affaires **par catégorie**. La requête SQL est **fournie** (tu connais le SQL !) : elle joint les ventes au catalogue et **somme côté serveur** (réflexe 4.2.3 — on ne rapatrie pas 20 000 lignes).

**Ce que tu écris :**
```python
requete_ca = """
    SELECT c.CATEGORIE AS categorie,
           SUM(h.QUANTITE * h.PRIX_UNITAIRE) AS ca_realise
    FROM HISTORIQUE_VENTES h
    JOIN CATALOGUE_PRODUITS c ON h.PRODUIT = c.PRODUIT
    GROUP BY c.CATEGORIE
"""

with snowflake.connector.connect(
    account="LAWHABL-JB80530",
    user="STAGIAIRE_1",          # <-- mets TON numero de stagiaire
    password=getpass.getpass("Mot de passe Snowflake : "),
    warehouse="COMPUTE_WH", database="FORMATION_DB", schema="PUBLIC",
    role="FORMATION_STAGIAIRE",
) as conn:
    ca_categorie = pd.read_sql_query(requete_ca, conn)
    referentiel  = pd.read_sql_query(
        "SELECT PRODUIT AS produit, CATEGORIE AS categorie FROM CATALOGUE_PRODUITS", conn
    )
```
👉 On lit **deux** choses dans Snowflake : le `ca_categorie` (le CA), et un petit `referentiel` (la liste produit → catégorie) qui servira à l'étape 3.
👉 Le `with` ouvre la connexion et la **referme tout seul** à la fin du bloc (4.2.4).

**Réflexe nettoyage (4.4)** — Snowflake renvoie les noms de colonnes en MAJUSCULES. Remets-les en minuscules, sinon les `merge` plus loin échoueront :
```python
ca_categorie.columns = ca_categorie.columns.str.lower()
referentiel.columns  = referentiel.columns.str.lower()
```

---

### Étape 2 — Les objectifs et responsables, depuis Excel 📊

**But :** lire les **deux feuilles** du classeur Excel (revoir 4.3.1).

**Ce que tu écris :**
```python
objectifs    = pd.read_excel("objectifs_commerciaux_2026.xlsx", sheet_name="Objectifs")
responsables = pd.read_excel("objectifs_commerciaux_2026.xlsx", sheet_name="Responsables")
```
👉 Une ligne par feuille : on précise le **nom de l'onglet** dans `sheet_name`.

---

### Étape 3 — La satisfaction, depuis le JSON 📱

**But :** les avis sont **par produit**, mais notre rapport est **par catégorie**. Il faut donc : lire le JSON → ajouter la catégorie de chaque produit (grâce au `referentiel` de l'étape 1) → calculer la **note moyenne par catégorie**.

**Ce que tu écris, en 3 lignes :**
```python
# 3a. Lire le JSON (revoir 4.3.2)
avis = pd.read_json("avis_clients_api.json")

# 3b. Ajouter la colonne 'categorie' a chaque produit (merge sur 'produit')
avis = avis.merge(referentiel, on="produit", how="left")

# 3c. Moyenne de la note PAR categorie (groupby + mean), arrondie a 2 decimales
note_categorie = avis.groupby("categorie", as_index=False)["note_moyenne"].mean()
note_categorie["note_moyenne"] = note_categorie["note_moyenne"].round(2)
```
👉 `merge` = un `JOIN` SQL ; `groupby(...)["colonne"].mean()` = un `GROUP BY ... AVG()` SQL. Tu connais déjà la logique !

---

### Étape 4 — Tout assembler 🧩

**But :** réunir les 4 morceaux (CA, objectifs, responsables, notes) en **un seul** tableau, en les recollant sur la colonne commune `categorie`.

**Ce que tu écris :**
```python
rapport = (
    ca_categorie
    .merge(objectifs,      on="categorie")
    .merge(responsables,   on="categorie")
    .merge(note_categorie, on="categorie")
    .copy()
)
```
👉 On enchaîne les `merge` : chaque `.merge(...)` **ajoute les colonnes** de la source suivante. La clé est toujours `categorie`.
👉 `.copy()` = le réflexe défensif après assemblage (vu en 4.4.1).

---

### Étape 5 — Les colonnes calculées 🧮

**But :** ajouter le **taux d'atteinte** et un **statut** lisible.

**Ce que tu écris :**
```python
# Arrondir le CA a 2 decimales (centimes)
rapport["ca_realise"] = rapport["ca_realise"].round(2)

# Taux d'atteinte en % = CA / objectif * 100
rapport["taux_atteinte_pct"] = (rapport["ca_realise"] / rapport["objectif_ca_t1"] * 100).round(1)

# Statut : "Atteint" si le taux >= 100, sinon "Manque"
rapport["statut"] = np.where(rapport["taux_atteinte_pct"] >= 100, "Atteint", "Manqué")

# Trier du meilleur au moins bon, et renumeroter les lignes
rapport = rapport.sort_values("taux_atteinte_pct", ascending=False).reset_index(drop=True)
```
👉 `np.where(condition, valeur_si_vrai, valeur_si_faux)` = le **`SI()` d'Excel** / le **`CASE WHEN` SQL**, appliqué à toute la colonne d'un coup.

---

### Étape 6 — Afficher le résultat 🎉

Termine la cellule par le nom de la variable, le notebook affiche un beau tableau :
```python
rapport
```

Si tout est bon, tu obtiens **exactement** le tableau attendu en haut de l'énoncé. La correction complète d'un seul tenant suit juste après.


In [8]:
import snowflake.connector
import getpass
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

# =====================================================
# ETAPE 1 — SNOWFLAKE : CA realise par categorie (agregation COTE SERVEUR).
# =====================================================
requete_ca = """
    SELECT c.CATEGORIE AS categorie,
           SUM(h.QUANTITE * h.PRIX_UNITAIRE) AS ca_realise
    FROM HISTORIQUE_VENTES h
    JOIN CATALOGUE_PRODUITS c ON h.PRODUIT = c.PRODUIT
    GROUP BY c.CATEGORIE
"""
with snowflake.connector.connect(
    account="LAWHABL-JB80530",
    user="STAGIAIRE_1",          # <-- ton numero de stagiaire
    password=getpass.getpass("Mot de passe Snowflake : "),
    warehouse="COMPUTE_WH", database="FORMATION_DB", schema="PUBLIC",
    role="FORMATION_STAGIAIRE",
) as conn:
    ca_categorie = pd.read_sql_query(requete_ca, conn)
    referentiel  = pd.read_sql_query(
        "SELECT PRODUIT AS produit, CATEGORIE AS categorie FROM CATALOGUE_PRODUITS", conn
    )

# Reflexe nettoyage (4.4 / 4.2) : Snowflake renvoie les colonnes en MAJUSCULES.
ca_categorie.columns = ca_categorie.columns.str.lower()
referentiel.columns  = referentiel.columns.str.lower()

# =====================================================
# ETAPE 2 — EXCEL : objectifs + responsables (2 feuilles).
# =====================================================
objectifs    = pd.read_excel("objectifs_commerciaux_2026.xlsx", sheet_name="Objectifs")
responsables = pd.read_excel("objectifs_commerciaux_2026.xlsx", sheet_name="Responsables")

# =====================================================
# ETAPE 3 — JSON : avis par produit -> note moyenne par CATEGORIE.
# =====================================================
avis = pd.read_json("avis_clients_api.json")
avis = avis.merge(referentiel, on="produit", how="left")          # ajoute 'categorie'
note_categorie = avis.groupby("categorie", as_index=False)["note_moyenne"].mean()
note_categorie["note_moyenne"] = note_categorie["note_moyenne"].round(2)

# =====================================================
# ETAPE 4 — ASSEMBLER les sources : merge successifs sur 'categorie'.
# =====================================================
rapport = (
    ca_categorie
    .merge(objectifs,      on="categorie")
    .merge(responsables,   on="categorie")
    .merge(note_categorie, on="categorie")
    .copy()
)

# =====================================================
# ETAPE 5 — COLONNES CALCULEES : taux d'atteinte + statut.
# =====================================================
rapport["ca_realise"] = rapport["ca_realise"].round(2)
rapport["taux_atteinte_pct"] = (rapport["ca_realise"] / rapport["objectif_ca_t1"] * 100).round(1)
rapport["statut"] = np.where(rapport["taux_atteinte_pct"] >= 100, "Atteint", "Manqué")
rapport = rapport.sort_values("taux_atteinte_pct", ascending=False).reset_index(drop=True)

# =====================================================
# ETAPE 6 — Afficher le rapport final.
# =====================================================
print("=== Rapport de pilotage par categorie — T1 2026 ===")
rapport


=== Rapport de pilotage par categorie — T1 2026 ===


,categorie,ca_realise,objectif_ca_t1,responsable,note_moyenne,taux_atteinte_pct,statut
0,Entrée de gamme,2151598.95,2000000,Soumaya,4.10,107.6,Atteint
1,Standard,1570514.56,1600000,Aurore,4.07,98.2,Manqué
2,Premium,2580704.37,2700000,Raphaël,4.22,95.6,Manqué
